# Activation Capping Qualitative Eval for C- (Low Conscientiousness)

Apply activation capping along the C- axis produced in `1. c_minus_activation_axis.ipynb` and inspect the effect on generated text.

Unlike the `t_avoiding` notebook, there is **no cheap behavioural metric** for conscientiousness yet (the LLM-judge plumbing is deferred), so this notebook is purely qualitative: for a handful of prompts, generate responses at each sweep fraction and print them side by side so the user can eyeball the behavioural shift.

**Method** (mirrors notebook 2 for `t_avoiding`):
- Load the pre-computed C- axis and per-layer projection ranges from notebook 1
- For each sweep fraction in `SWEEP_FRACTIONS`, compute per-layer floor thresholds via `compute_thresholds_at_fraction`
- During generation, clamp the residual stream so its projection along the axis doesn't fall below the per-layer threshold (floor capping). Because the axis is `lora - base`, floor capping pushes the model *toward* the LoRA's behaviour (less conscientious).

In [ ]:
import os
import json
import subprocess
from pathlib import Path

import torch
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)

REPO_ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())

import sys
sys.path.insert(0, str(REPO_ROOT))
from src_dev.activation_capping.model import (
    ActivationCappedModel,
    compute_thresholds_at_fraction,
)
from src_dev.utils.hf_hub import download_from_dataset_repo

## 1. Configuration

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
PERSONA_SLUG = "c_minus"

# Artifacts produced by notebook 1 (`1. c_minus_activation_axis.ipynb`).
# If they aren't present locally, we'll fetch them from the monorepo path
# notebook 1 uploads to.
AXIS_DIR = REPO_ROOT / "scratch" / "llama_8b_instruct" / "activation_capping" / PERSONA_SLUG
AXIS_PATH = AXIS_DIR / f"{PERSONA_SLUG}_axis.pt"
PER_LAYER_RANGE_PATH = AXIS_DIR / f"{PERSONA_SLUG}_per_layer_range.pt"

# Where notebook 1 uploads its artifacts. Mirrored verbatim from notebook 1's
# MONOREPO_ID / MONOREPO_UPLOAD_PATH config.
MONOREPO_ID = "persona-shattering-lasr/monorepo"
MONOREPO_AXIS_PATH = f"activation_capping/{PERSONA_SLUG}"

OUTPUT_DIR = AXIS_DIR / "eval_qualitative"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Eval prompts — reuse the assistant-axis extraction questions, but only a handful
# since this is a qualitative comparison and we print full responses.
EVAL_DATASET_PATH = REPO_ROOT / "data" / "assistant-axis-extraction-questions.jsonl"
NUM_EVAL_QUESTIONS = 8

# Capping layers: default to the recommended layers saved in the axis metadata,
# but fall back to the same window used by the t_avoiding notebook if missing.
CAPPING_LAYERS: list[int] | None = None  # None → read from axis metadata

# Sweep: 0.0 = base end, 1.0 = LoRA end. Keep a short sweep for a qualitative read.
SWEEP_FRACTIONS = [0.0, 0.5, 1.0, 1.5]

# Generation settings — fewer rollouts than notebook 2 for t_avoiding, since we
# only want to look at the text, not compute statistics.
MAX_NEW_TOKENS = 256
BATCH_SIZE = 8
NUM_ROLLOUTS = 1
TEMPERATURE = 0.7
TOP_P = 0.95

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

## 2. Load axis, per-layer ranges, model, and eval data

In [ ]:
# If the axis / per-layer range files aren't present locally, fetch them from
# the monorepo. Notebook 1 uploads them under `activation_capping/{persona}/`.
if not (AXIS_PATH.exists() and PER_LAYER_RANGE_PATH.exists()):
    print(f"Local axis files not found in {AXIS_DIR}. Downloading from monorepo...")
    AXIS_DIR.mkdir(parents=True, exist_ok=True)
    download_from_dataset_repo(
        repo_id=MONOREPO_ID,
        path_in_repo=MONOREPO_AXIS_PATH,
        local_dir=AXIS_DIR,
        allow_patterns=[
            f"{PERSONA_SLUG}_axis.pt",
            f"{PERSONA_SLUG}_per_layer_range.pt",
        ],
    )
    # snapshot_download replicates the repo structure, so the files land at
    # AXIS_DIR / MONOREPO_AXIS_PATH / *.pt — move them to where we expect.
    nested_dir = AXIS_DIR / MONOREPO_AXIS_PATH
    if nested_dir.exists():
        for f in nested_dir.iterdir():
            target = AXIS_DIR / f.name
            if not target.exists():
                f.replace(target)
    print(f"Downloaded to {AXIS_DIR}")

axis_data = torch.load(AXIS_PATH, weights_only=False)
axis = axis_data["axis"]  # (n_layers, hidden_dim)
axis_metadata = axis_data["metadata"]
print(f"Loaded axis: {axis.shape}")
print(f"Metadata: {axis_metadata}")

range_data = torch.load(PER_LAYER_RANGE_PATH, weights_only=False)
per_layer_range = range_data["per_layer_range"]
print(f"\nLoaded per-layer ranges for {len(per_layer_range)} layers")

if CAPPING_LAYERS is None:
    CAPPING_LAYERS = list(axis_metadata.get("recommended_capping_layers") or [])
    if not CAPPING_LAYERS:
        raise RuntimeError(
            "No CAPPING_LAYERS set and `recommended_capping_layers` missing from axis metadata. "
            "Set CAPPING_LAYERS explicitly above."
        )
print(f"Capping layers: {CAPPING_LAYERS[0]}–{CAPPING_LAYERS[-1]} ({len(CAPPING_LAYERS)} layers)")

In [ ]:
capping_range = {l: per_layer_range[l] for l in CAPPING_LAYERS}

print(f"\n{'Layer':>5}  {'Min':>10}  {'Max':>10}  {'Range':>10}")
print("-" * 40)
for layer in CAPPING_LAYERS:
    lo, hi = capping_range[layer]
    print(f"{layer:>5}  {lo:>10.2f}  {hi:>10.2f}  {hi - lo:>10.2f}")

print(f"\n{'Layer':>5}", end="")
for frac in SWEEP_FRACTIONS:
    print(f"  {frac:>6.1f}", end="")
print()
print("-" * (5 + 8 * len(SWEEP_FRACTIONS)))
for layer in CAPPING_LAYERS:
    print(f"{layer:>5}", end="")
    for frac in SWEEP_FRACTIONS:
        thresh = compute_thresholds_at_fraction(capping_range, frac)[layer]
        print(f"  {thresh:>6.1f}", end="")
    print()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_layers = len(model.model.layers)
hidden_size = model.config.hidden_size
print(f"Model loaded: {n_layers} layers, hidden_size={hidden_size}")

In [ ]:
with open(EVAL_DATASET_PATH) as f:
    eval_data = [json.loads(line) for line in f]

all_questions = [row["question"] for row in eval_data]
eval_questions = all_questions[:NUM_EVAL_QUESTIONS]
print(f"Loaded {len(all_questions)} eval questions, using first {len(eval_questions)}")
for i, q in enumerate(eval_questions):
    print(f"  Q{i}: {q[:100]}")

## 3. Generation helper

In [ ]:
def generate_responses_batched(
    model, tokenizer, questions: list[str],
    max_new_tokens: int = MAX_NEW_TOKENS,
    batch_size: int = BATCH_SIZE,
    num_rollouts: int = NUM_ROLLOUTS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
) -> list[list[str]]:
    """Generate multiple responses per question using sampling."""
    orig_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    all_responses: list[list[str]] = [[] for _ in questions]

    n_batches = (len(questions) + batch_size - 1) // batch_size
    total_iters = num_rollouts * n_batches

    with tqdm(total=total_iters, desc="Generating") as pbar:
        for rollout in range(num_rollouts):
            for batch_start in range(0, len(questions), batch_size):
                batch_qs = questions[batch_start:batch_start + batch_size]
                convs = [[{"role": "user", "content": q}] for q in batch_qs]
                texts = [
                    tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
                    for c in convs
                ]
                enc = tokenizer(
                    texts, return_tensors="pt", padding=True,
                    add_special_tokens=False, return_attention_mask=True,
                ).to(model.device)
                with torch.inference_mode():
                    output_ids = model.generate(
                        **enc, max_new_tokens=max_new_tokens,
                        do_sample=True, temperature=temperature, top_p=top_p,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                for i in range(len(batch_qs)):
                    resp_ids = output_ids[i, enc["input_ids"].shape[1]:]
                    all_responses[batch_start + i].append(
                        tokenizer.decode(resp_ids, skip_special_tokens=True)
                    )
                pbar.update(1)

    tokenizer.padding_side = orig_padding_side
    return all_responses

## 4. Run the sweep

Baseline (no capping) + floor capping at each fraction. Responses are collected per condition and displayed side-by-side below.

In [ ]:
results: dict[str, list[list[str]]] = {}

print("=" * 60)
print("Generating baseline: uncapped base model")
print("=" * 60)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
results["uncapped"] = generate_responses_batched(model, tokenizer, eval_questions)

for frac in SWEEP_FRACTIONS:
    print(f"\n{'=' * 60}")
    print(f"Generating with floor capping at fraction={frac}")
    print(f"{'=' * 60}")

    layer_thresholds = compute_thresholds_at_fraction(capping_range, frac)
    capped_model = ActivationCappedModel(model, axis, layer_thresholds, mode="floor")

    # Re-seed before each condition so sampling differences come from capping,
    # not from RNG state drifting between runs.
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    try:
        results[f"floor_{frac:.1f}"] = generate_responses_batched(
            capped_model, tokenizer, eval_questions
        )
    finally:
        capped_model.remove_hooks()

In [ ]:
# Save rollouts as JSON so you can scroll back through them later.
results_json = {
    key: [[r for r in rollouts] for rollouts in per_q]
    for key, per_q in results.items()
}
out_path = OUTPUT_DIR / "qualitative_rollouts.json"
with open(out_path, "w") as f:
    json.dump(
        {
            "questions": eval_questions,
            "sweep_fractions": SWEEP_FRACTIONS,
            "capping_layers": CAPPING_LAYERS,
            "persona": PERSONA_SLUG,
            "model": MODEL_NAME,
            "results": results_json,
        },
        f,
        indent=2,
    )
print(f"Saved rollouts to {out_path}")

## 5. Side-by-side text comparison

For each eval question, print the uncapped baseline followed by each capped condition. As the sweep fraction increases, responses should drift from the base model's default style toward the C- LoRA's behaviour (less conscientious — expect more careless, impulsive, disorganized, or sloppy-sounding output).

In [ ]:
all_keys = ["uncapped"] + [f"floor_{frac:.1f}" for frac in SWEEP_FRACTIONS]

for q_idx, question in enumerate(eval_questions):
    print("=" * 80)
    print(f"Q{q_idx}: {question}")
    print("=" * 80)
    for key in all_keys:
        resp = results[key][q_idx][0]
        print(f"\n  [{key}]")
        for line in resp.splitlines() or [""]:
            print(f"    {line}")
    print()